# Trajectory Analysis: RMSF Colouring, Region Annotation, and Movie Export

This tutorial walks through a typical trajectory analysis workflow:

1. Load a multi-frame molecular system (here an NMR ensemble, but the same steps apply to MD trajectories).
2. Compute per-residue RMSF and map it onto the viewer as a colour gradient.
3. Annotate conformational subsets as named **regions**.
4. Export the annotated scene as an interactive HTML file and as a video.

**Requirements:** `molsysmt`, `molsysviewer`, `molsysviewer-molsysmt`.

In [ ]:
import molsysmt as msm
import molsysviewer as msv
from molsysviewer_molsysmt import get_addon, lifecycle, on_enable
from molsysviewer_molsysmt.runtime import ensure_runtime

In [ ]:
msv.addons.register(get_addon(), lifecycle=lifecycle)

## Load a multi-frame system

We use the TRP-cage miniprotein NMR ensemble (PDB 1L2Y, 38 conformers) as a
convenient stand-in for an MD trajectory.  `molsysmt.MolSys` stores all frames
in memory; `view.load()` displays the first frame by default.

In [ ]:
ms = msm.convert("pdb_id:1l2y", to_form="molsysmt.MolSys")
n_frames = msm.get(ms, element="system", n_structures=True)
print(f"Frames loaded: {n_frames}")

## Create the view and attach the system to the addon runtime

In [ ]:
view = msv.MolSysView()
view.load(ms)

runtime = ensure_runtime(view)
runtime.molecular_system = ms

on_enable(view)
view.show()

## Colour by RMSF

Root-mean-square fluctuation (RMSF) quantifies how much each atom moves across
frames relative to the mean structure.  High-RMSF regions (flexible loops) appear
in warm colours; rigid secondary-structure elements appear in cool colours.

In [ ]:
color_panel = view.addons.resolve_panel_widget("molsysmt", "color")
color_panel.handle_action(
    view,
    "apply_color",
    {"property": "rmsf", "palette": "coolwarm"},
)

## Annotate conformational subsets as regions

MolSysViewer **regions** are named atom subsets that can be toggled, coloured,
and exported independently.  Here we create two regions that highlight the
flexible N-terminal tail (residues 1–3) and the rigid hydrophobic core.

In [ ]:
# Select flexible N-terminal residues
flex_indices = msm.select(ms, selection="residue_index in [0, 1, 2]", element="atom")
view.regions.add(flex_indices, name="flexible-tail", color="#e74c3c")

# Select the hydrophobic core (Trp6, Pro12, Pro17, Pro18, Pro19)
core_indices = msm.select(
    ms,
    selection="group_name in ['TRP', 'PRO'] and residue_index in [5, 11, 16, 17, 18]",
    element="atom",
)
view.regions.add(core_indices, name="hydrophobic-core", color="#2980b9")

## Configure and export a movie

`view.movie` builds an animation by cycling through the trajectory frames.
The exported GIF or MP4 captures the RMSF colouring and region overlays.

In [ ]:
view.movie.set_timeline(
    structure_indices=list(range(n_frames)),
    fps=5,
)

# Export as animated GIF (requires imageio)
view.movie.export("1l2y_rmsf.gif")

## Export an interactive HTML snapshot

In [ ]:
view.export.html("1l2y_trajectory.html", title="TRP-cage NMR ensemble — RMSF")

## Next steps

- Replace the NMR ensemble with an MD trajectory loaded via `msm.convert("my_traj.xtc", ...)`.
- Use the **Structure** panel to compute RMSD across frames and identify dominant conformers.
- See `tutorial_topomt_pocket.ipynb` to add a pocket analysis layer on top of the trajectory.